<a href="https://colab.research.google.com/github/Anshulmohan27/Machine_Learning/blob/main/Titanic_Survival_Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
#

In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [54]:
titanic_train = pd.read_csv('train.csv')
titanic_test = pd.read_csv('test.csv')

titanic_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [55]:
titanic_train_y = titanic_train.Survived
titanic_train_X = titanic_train.drop(['Survived'], axis=1)

In [57]:
def feature_eng(df):
  df = df.drop(columns = 'PassengerId')
  df = df.drop(columns = 'Name')
  df = df.drop(columns = 'Ticket')
  df = df.drop(columns = 'Cabin')
  df['IsAlone'] = (df['SibSp'] + df['Parch']).apply(lambda x: 0 if x>0 else 1)
  return df

titanic_train_X = feature_eng(titanic_train_X)
titanic_test = feature_eng(titanic_test)
titanic_train_X.head()


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,IsAlone
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,0
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,0
4,3,male,35.0,0,0,8.0500,S,1


In [58]:
titanic_test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,IsAlone
0,3,male,34.5,0,0,7.8292,Q,1
1,3,female,47.0,1,0,7.0000,S,0
2,2,male,62.0,0,0,9.6875,Q,1
3,3,male,27.0,0,0,8.6625,S,1
4,3,female,22.0,1,1,12.2875,S,0


In [59]:
titanic_train_X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    object 
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Embarked  889 non-null    object 
 7   IsAlone   891 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB


In [60]:
titanic_train_X['Age'] = titanic_train_X['Age'].fillna(titanic_train_X['Age'].median())
titanic_train_X['Embarked'] = titanic_train_X['Embarked'].fillna(titanic_train_X['Embarked'].mode()[0])

In [61]:
titanic_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    418 non-null    int64  
 1   Sex       418 non-null    object 
 2   Age       332 non-null    float64
 3   SibSp     418 non-null    int64  
 4   Parch     418 non-null    int64  
 5   Fare      417 non-null    float64
 6   Embarked  418 non-null    object 
 7   IsAlone   418 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 26.3+ KB


In [62]:
titanic_test['Age'] = titanic_test['Age'].fillna(titanic_test['Age'].median())
titanic_test['Fare'] = titanic_test['Fare'].fillna(titanic_test['Fare'].median())


In [63]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

t_train_encoded = encoder.fit_transform(titanic_train_X[['Sex', 'Embarked']])
t_test_encoded = encoder.transform(titanic_test[['Sex', 'Embarked']])

t_train_encoded = pd.DataFrame(t_train_encoded, columns=encoder.get_feature_names_out(['Sex', 'Embarked']))
t_test_encoded = pd.DataFrame(t_test_encoded, columns=encoder.get_feature_names_out(['Sex', 'Embarked']))



In [64]:
t_train_encoded.head()

,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.0,1.0,0.0,0.0,1.0
1,1.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,1.0


In [65]:
t_test_encoded.head()


,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.0,1.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,1.0,0.0
3,0.0,1.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,1.0


In [66]:
titanic_train_X = titanic_train_X.drop(columns = ['Sex', 'Embarked'])
titanic_test = titanic_test.drop(columns = ['Sex', 'Embarked'])

In [67]:
titanic_train_X = pd.concat([titanic_train_X, t_train_encoded], axis=1)
titanic_test = pd.concat([titanic_test, t_test_encoded], axis=1)

In [68]:
titanic_train_X.head()

,Pclass,Age,SibSp,Parch,Fare,IsAlone,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,3,22.0,1,0,7.2500,0,0.0,1.0,0.0,0.0,1.0
1,1,38.0,1,0,71.2833,0,1.0,0.0,1.0,0.0,0.0
2,3,26.0,0,0,7.9250,1,1.0,0.0,0.0,0.0,1.0
3,1,35.0,1,0,53.1000,0,1.0,0.0,0.0,0.0,1.0
4,3,35.0,0,0,8.0500,1,0.0,1.0,0.0,0.0,1.0


In [69]:
titanic_test.head()

,Pclass,Age,SibSp,Parch,Fare,IsAlone,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,3,34.5,0,0,7.8292,1,0.0,1.0,0.0,1.0,0.0
1,3,47.0,1,0,7.0000,0,1.0,0.0,0.0,0.0,1.0
2,2,62.0,0,0,9.6875,1,0.0,1.0,0.0,1.0,0.0
3,3,27.0,0,0,8.6625,1,0.0,1.0,0.0,0.0,1.0
4,3,22.0,1,1,12.2875,0,1.0,0.0,0.0,0.0,1.0


In [109]:
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(titanic_train_X, titanic_train_y)
predictions = model.predict(titanic_test)
final_predictions = (predictions > 0.5).astype(int)


In [110]:
test = pd.read_csv('test.csv')
t_test_PassId = test['PassengerId']
prediction = pd.DataFrame({
    "PassengerId": test["PassengerId"].values,
    "Survived": final_predictions
})
prediction.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,1
3,895,1
4,896,1


In [111]:
prediction['Survived'].mean()

np.float64(0.38516746411483255)

In [99]:
prediction.to_csv("Random_forest_submission.csv", index= False)